# 30/ Anomaly Detection System

**Dataset (Kaggle):** [https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

> Same transactions dataset as Project 18, used here to demonstrate *unsupervised* anomaly detection (no labels used during training).

**Goal:** Detect rare/anomalous transactions without relying on labels, using Isolation Forest and One-Class SVM.

**How to get the data:**
1. Download from the link above (or run `kaggle datasets download -d <dataset-slug> -p data/ --unzip`, requires a `kaggle.json` API token in `~/.kaggle/`).
2. Place the extracted file(s) in a local `data/` folder next to this notebook.
3. Update `DATA_PATH` in the cell below if your filename differs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA

sns.set_style("whitegrid")


In [ ]:
DATA_PATH = "data/creditcard.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df['Class'].value_counts(normalize=True))
df.head()


## EDA: Class Imbalance

In [ ]:
sns.countplot(x='Class', data=df)
plt.yscale('log')
plt.title("Class distribution (log scale) - 1 = anomaly")
plt.show()


## Preprocessing

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Contamination = expected proportion of anomalies in the data
contamination = y.mean()
print("Estimated contamination:", round(contamination, 5))


## Unsupervised Anomaly Detection: Isolation Forest

In [ ]:
iso_forest = IsolationForest(n_estimators=200, contamination=contamination, random_state=42, n_jobs=-1)
iso_forest.fit(X_scaled)

# IsolationForest outputs -1 for anomalies, 1 for normal; convert to 0/1 to match 'Class'
preds = iso_forest.predict(X_scaled)
preds = np.where(preds == -1, 1, 0)

print(classification_report(y, preds))
cm = confusion_matrix(y, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Isolation Forest - Confusion Matrix")
plt.show()


## Visualize Anomalies (PCA projection)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7,6))
plt.scatter(X_pca[preds==0,0], X_pca[preds==0,1], s=3, alpha=0.3, label='Normal')
plt.scatter(X_pca[preds==1,0], X_pca[preds==1,1], s=8, color='red', label='Anomaly')
plt.legend()
plt.title("Isolation Forest anomalies (PCA projection)")
plt.show()


## Alternative: One-Class SVM (on a data sample for speed)

In [ ]:
sample_idx = np.random.RandomState(42).choice(len(X_scaled), size=20000, replace=False)
X_sample, y_sample = X_scaled[sample_idx], y.values[sample_idx]

ocsvm = OneClassSVM(nu=contamination, kernel='rbf', gamma='scale')
ocsvm.fit(X_sample)
preds_svm = ocsvm.predict(X_sample)
preds_svm = np.where(preds_svm == -1, 1, 0)
print(classification_report(y_sample, preds_svm))


## Next Steps
- Compare against a supervised baseline (see Project 18) since labels exist here.
- Try an autoencoder: flag high reconstruction error as anomalous.
- Tune `contamination` / `nu` against a validation set rather than the true (usually unknown) rate.